In [10]:
"""
Transform a fully type-resolved triples file into Neo4j-ready CSVs: one
nodes_<Label>.csv per taxonomy label (deduplicated), and one
relationships.csv with evidence properties.

Expects source_type/target_type to already be clean -- each entity name
should map to exactly one type. If you had entity-type ambiguity, run
find_entity_type_ambiguities.py, then apply_manual_llm_resolution.py or
llm_resolve_entity_types.py, then apply_type_lookup_to_triples.py first, and
point TRIPLES_PATH below at that last script's output. This script has no
knowledge of the lookup table at all -- it just trusts source_type/
target_type as given.

Designed for Jupyter/Colab execution. No __main__ guard -- just set
TRIPLES_PATH below and run the whole cell/file.
"""

import re
import hashlib
import os
from collections import Counter, defaultdict

import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = r"..\STEP-6-OPTIONAL-CLEAN-UP-STEPS\Resolving-Entity-Ambuiguity\triples_types_resolved.xlsx"   # <- output of apply_type_lookup_to_triples.py
OUT_DIR = "neo4j_import"

SOURCE_COL = "resolved_source"
TARGET_COL = "resolved_target"
SOURCE_TYPE_COL = "source_type"
TARGET_TYPE_COL = "target_type"
INTERACTION_COL = "interaction"
CATEGORY_COL = "category"   # what kind of comparison this row is, e.g. treatment_property_change

TAXONOMY_TO_LABEL = {
    "functional_property": "FunctionalProperty",
    "physicochemical_property": "PhysicochemicalProperty",
    "structural_property": "StructuralProperty",
    "modification_method": "ModificationMethod",
    "extraction_method": "ExtractionMethod",
    "material_used_directly": "MaterialUsedDirectly",
    "rheological_property": "RheologicalProperty",
    "sensory_property": "SensoryProperty",
}


def clean_text(val):
    """Flatten to a single line, safe for CSV/LOAD CSV; empty string for NaN."""
    if pd.isna(val):
        return ""
    s = str(val)
    s = s.replace("\r\n", " ").replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def to_rel_type(interaction):
    """increased -> INCREASED, no_significant_change -> NO_SIGNIFICANT_CHANGE"""
    s = clean_text(interaction).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s.upper() if s else "OTHER"


# ---------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------
os.makedirs(OUT_DIR, exist_ok=True)

if TRIPLES_PATH.lower().endswith(".csv"):
    df = pd.read_csv(TRIPLES_PATH)
else:
    df = pd.read_excel(TRIPLES_PATH)
print(f"Loaded {len(df)} triples from {TRIPLES_PATH}")

seen_types = set(df[SOURCE_TYPE_COL].dropna().unique()) | set(df[TARGET_TYPE_COL].dropna().unique())
unexpected = seen_types - set(TAXONOMY_TO_LABEL)
if unexpected:
    raise SystemExit(f"STOP: unexpected type values not in the closed taxonomy: {unexpected}")

# ---------------------------------------------------------------
# SANITY CHECK: every entity name should map to exactly one type by now.
# If not, this is a safety net (majority vote + a loud warning) rather than
# a silent wrong answer -- but the fix is to run the resolution scripts
# upstream, not to rely on this.
# ---------------------------------------------------------------
name_type_counts = defaultdict(Counter)
for _, r in df[[SOURCE_COL, SOURCE_TYPE_COL]].dropna().iterrows():
    name_type_counts[r[SOURCE_COL]][r[SOURCE_TYPE_COL]] += 1
for _, r in df[[TARGET_COL, TARGET_TYPE_COL]].dropna().iterrows():
    name_type_counts[r[TARGET_COL]][r[TARGET_TYPE_COL]] += 1

canonical_type = {name: counter.most_common(1)[0][0] for name, counter in name_type_counts.items()}
still_ambiguous = {name: counter for name, counter in name_type_counts.items() if len(counter) > 1}
if still_ambiguous:
    print(f"WARNING: {len(still_ambiguous)} entity names still map to more than one type "
          f"-- falling back to majority vote for those. Run "
          f"apply_type_lookup_to_triples.py first to avoid this.")

# ---------------------------------------------------------------
# BUILD NODE TABLES (one CSV per label), deduplicated
# ---------------------------------------------------------------
name_occurrences = Counter()
for col in [SOURCE_COL, TARGET_COL]:
    name_occurrences.update(df[col].dropna())

nodes_by_label = defaultdict(list)
for name, occ_count in name_occurrences.items():
    label = TAXONOMY_TO_LABEL[canonical_type[name]]
    nodes_by_label[label].append({"name": clean_text(name), "occurrence_count": occ_count})

total_nodes = 0
for label, rows in nodes_by_label.items():
    node_df = pd.DataFrame(rows).drop_duplicates(subset="name").sort_values("name")
    # encoding="utf-8-sig" adds a UTF-8 byte-order-mark. Without it, a plain
    # UTF-8 CSV opened directly in Excel gets misread as cp1252/ANSI and
    # Greek letters like alpha turn into mojibake (alpha -> "I-hat-plusminus")
    # -- this is what happened before. The BOM makes Excel autodetect UTF-8
    # correctly; every other tool (pandas, Neo4j's LOAD CSV) handles it fine.
    node_df.to_csv(os.path.join(OUT_DIR, f"nodes_{label}.csv"), index=False, encoding="utf-8-sig")
    total_nodes += len(node_df)
    print(f"  nodes_{label}.csv: {len(node_df)} nodes")
print(f"Total nodes across all labels: {total_nodes}")

# ---------------------------------------------------------------
# BUILD RELATIONSHIPS TABLE
# ---------------------------------------------------------------
rel_records = []
for idx, row in df.iterrows():
    src = clean_text(row[SOURCE_COL])
    tgt = clean_text(row[TARGET_COL])
    if not src or not tgt:
        continue
    rel_type = to_rel_type(row[INTERACTION_COL])
    raw = f"{src}|{tgt}|{rel_type}|{row.get('doi','')}|{row.get('chunk_id','')}|{idx}"
    evidence_id = hashlib.md5(raw.encode("utf-8")).hexdigest()[:12]
    rel_records.append({
        "evidence_id": evidence_id,
        "source_name": src,
        "target_name": tgt,
        "interaction_type": rel_type,
        "doi": clean_text(row.get("doi")),
        "chunk_id": clean_text(row.get("chunk_id")),
        "section": clean_text(row.get("section")),
        "pmid": clean_text(row.get("pmid")),
        "compared_property": clean_text(row.get("compared_property")),
        "reported_value": clean_text(row.get("reported_value")),
        "claim_status": clean_text(row.get("claim_status")),
        "category": clean_text(row.get(CATEGORY_COL)),
        "corresponding_sentence": clean_text(row.get("corresponding_sentence")),
    })

rel_df = pd.DataFrame(rel_records)
rel_df.to_csv(os.path.join(OUT_DIR, "relationships.csv"), index=False, encoding="utf-8-sig")
print(f"relationships.csv: {len(rel_df)} edges")

dropped = len(df) - len(rel_df)
if dropped:
    print(f"NOTE: {dropped} rows dropped from relationships (missing source/target)")

Loaded 10224 triples from ..\STEP-6-OPTIONAL-CLEAN-UP-STEPS\Resolving-Entity-Ambuiguity\triples_types_resolved.xlsx
  nodes_ModificationMethod.csv: 347 nodes
  nodes_MaterialUsedDirectly.csv: 709 nodes
  nodes_ExtractionMethod.csv: 128 nodes
  nodes_StructuralProperty.csv: 595 nodes
  nodes_FunctionalProperty.csv: 318 nodes
  nodes_PhysicochemicalProperty.csv: 359 nodes
  nodes_RheologicalProperty.csv: 134 nodes
  nodes_SensoryProperty.csv: 123 nodes
Total nodes across all labels: 2713
relationships.csv: 10224 edges
